# Guess the Condition
---

**Guess the Condition** asks a simple question about an experiment: *can the experimental conditions be told apart by looking at the images?* You are shown images one at a time, without their condition, and name the condition you think each one comes from. The tool keeps your guesses and the time you took, and then tells you whether you did better than chance **across the biological repeats**, which conditions get confused with which, and how long each decision took.

It helps you to:

* judge whether a phenotype is visible and reproducible before investing in its quantification;
* find the conditions that look alike;
* get a first, unbiased feeling for how consistent the repeats are.

The same game can be played from **Napari** and **Fiji** (tools *Prepare the game*, *Play a round*, *Analyse the results*), from a script (see the API example notebook) and from this notebook. The results are kept in a folder with your name: you can stop at any time and resume later, from any of them.

Notebook created by [Guillaume Jacquemet](https://cellmig.org/).

## Prepare your images

One folder per condition, inside it one folder per biological repeat, inside it the TIFF images (`.tif` or `.tiff`):

```
Experiments
├── Control
│   ├── R1   (FOV1.tif, FOV2.tif, ...)
│   └── R2
└── Mutant
    ├── R1
    └── R2
```

The names of the condition and repeat folders are the *condition* and the *biological repeat* of every image inside. Multi-channel images are shown with the channels side by side; z-stacks and time series as a maximum projection.

# Step 0: Initialize environment & install dependencies
---

**Run the following cell to initialize the environment.**

* **For Google Colab:** the notebook installs Guess the Condition and mounts Google Drive.
* **For a local Jupyter Lab (the installed app):** nothing is installed: everything is already there.

In [ ]:
# @title Initialize environment & install dependencies
import sys

# Metadata
current_version = "0.1.0"
notebook_name = "GuessTheCondition"

if "google.colab" in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")
    !pip install -q "guessthecondition @ git+https://github.com/CellMigrationLab/GuessTheCondition"
    from google.colab import drive
    drive.mount("/content/gdrive")
    print("✅ Colab setup done")
else:
    print("✅ Local environment detected. Assuming dependencies are already installed.")

import ipywidgets as widgets
from IPython.display import Image, display, clear_output

import guessthecondition
from guessthecondition import GameError, Session, analyze, describe, load_for_display, read_results, scan_experiment

print("%s %s, guessthecondition %s" % (notebook_name, current_version, guessthecondition.__version__))

# Step 1: Choose your data and prepare the game
---
Give your name (your results are kept in a folder with that name inside the results folder, so you can resume later), the folder with your experiment, a folder for the results, and the share of the images you want to guess. The images are shown in an order spread over the conditions and the repeats. Ticking *Start again* keeps your earlier results as a backup.

In [ ]:
# @title Prepare the game
session = None

wide = widgets.Layout(width="600px")
style = {"description_width": "140px"}
w_name = widgets.Text(value="YourName", description="Your name:", layout=wide, style=style)
w_experiment = widgets.Text(value="", placeholder="/content/gdrive/MyDrive/Experiments", description="Experiment folder:", layout=wide, style=style)
w_results = widgets.Text(value="", placeholder="/content/gdrive/MyDrive/Results", description="Results folder:", layout=wide, style=style)
w_percentage = widgets.IntSlider(value=20, min=1, max=100, description="% to guess:", layout=wide, style=style)
w_restart = widgets.Checkbox(value=False, description="Start again (the earlier results are kept as a backup)", indent=False)
w_fix_seed = widgets.Checkbox(value=False, description="Fix the order of the images with a seed", indent=False)
w_seed = widgets.IntText(value=0, description="Seed:", style=style)
prepare_button = widgets.Button(description="Prepare the game", button_style="primary", layout=widgets.Layout(width="200px"))
prepare_out = widgets.Output()


def on_prepare(_):
    global session
    with prepare_out:
        clear_output()
        try:
            session = Session.create(
                w_experiment.value.strip(), w_results.value.strip(), w_name.value, w_percentage.value,
                seed=w_seed.value if w_fix_seed.value else None, restart=w_restart.value,
            )
        except GameError as error:
            session = None
            print("⚠️", error.message)
            return
        print(describe(scan_experiment(session.experiment_dir)))
        print()
        print("Conditions: " + session.conditions_text())
        print("Images to guess: %d (%d guessed so far, %d to go)" % (session.target, session.n_answered, session.remaining))
        print("Your game is in: %s" % session.folder)
        print("\n✅ Ready: go on to Step 2.")


prepare_button.on_click(on_prepare)
w_restart.value = False
display(widgets.VBox([w_name, w_experiment, w_results, w_percentage, w_restart, widgets.HBox([w_fix_seed, w_seed]), prepare_button, prepare_out]))

# Step 2: Guess the condition
---
Press **Start / continue** to see an image, choose the condition you think it comes from and press **Submit guess**. The next image appears at once. The time you take is recorded from the moment an image is shown. A guess made by mistake can be undone with **Undo last guess**. Stop whenever you like: your guesses are saved after every image, and pressing **Start / continue** later (here, in Napari or in Fiji) picks up where you stopped.

You are not told whether a guess was right until the analysis.

In [ ]:
# @title Play
import matplotlib.pyplot as plt

play_out = widgets.Output()
start_button = widgets.Button(description="Start / continue", button_style="primary")
submit_button = widgets.Button(description="Submit guess", button_style="success", disabled=True)
undo_button = widgets.Button(description="Undo last guess", disabled=True)
choice = widgets.ToggleButtons(options=[], description="Condition:", value=None)
progress_label = widgets.HTML("")


def show_image(item, channels):
    fig, axes = plt.subplots(1, len(channels), figsize=(5 * len(channels), 5), squeeze=False)
    for i, channel in enumerate(channels):
        axes[0][i].imshow(channel, cmap="gray", vmin=0, vmax=1)
        axes[0][i].set_title("Channel %d" % (i + 1) if len(channels) > 1 else "")
        axes[0][i].axis("off")
    fig.suptitle("Image %d of %d" % (item["number"], item["of"]))
    fig.tight_layout()
    plt.show()


def next_image():
    with play_out:
        clear_output(wait=True)
        try:
            item, channels, left_out = session.serve_readable(load_for_display)
            if left_out:
                print("⚠️ Left out (unreadable):", ", ".join(left_out))
        except GameError as error:
            submit_button.disabled = True
            undo_button.disabled = session.n_answered == 0
            progress_label.value = ""
            print("🎉" if error.code == "finished" else "⚠️", error.message)
            return
        submit_button.disabled = False
        undo_button.disabled = session.n_answered == 0
        progress_label.value = "<b>Image %d of %d</b>" % (item["number"], item["of"])
        show_image(item, channels)
        if item["fresh"]:
            session.start_clock()


def on_start(_):
    if session is None:
        with play_out:
            clear_output()
            print("⚠️ Prepare the game first (Step 1).")
        return
    choice.options = session.conditions
    choice.value = None
    next_image()


def on_submit(_):
    if choice.value is None:
        with play_out:
            print("Choose a condition first.")
        return
    session.answer(choice.value)
    choice.value = None
    next_image()


def on_undo(_):
    try:
        undone = session.undo_last()
    except GameError as error:
        with play_out:
            print("⚠️", error.message)
        return
    choice.value = None
    next_image()


start_button.on_click(on_start)
submit_button.on_click(on_submit)
undo_button.on_click(on_undo)
display(widgets.VBox([widgets.HBox([start_button, undo_button]), progress_label, choice, submit_button, play_out]))

# Step 3: Analyse the results
---
Run the next cell when you have guessed enough images. It reports, in this order:

1. **The answer**: can the conditions be told apart *across the biological repeats*? The repeat is the replicate, so the accuracy of every repeat is compared with chance (a one-sided t-test on the repeats' accuracy minus chance, from 3 repeats, with n and the 95% interval). Chance is the share of the most common condition. With one or two repeats there is no verdict about the experiment: a randomization test says whether you beat chance on these images (exploratory) and you are told what to change next time.
2. **Exploratory**: the test on all images pooled (exact binomial) and a test of whether the repeats differ. They count every image as independent, so their false-positive chance is higher than stated.
3. Accuracy per repeat and per condition, the confusion matrix and the pairs of conditions that are mistaken for each other, and the decision times.

**Chance** is the accuracy of someone who ignores the images and always names the most common condition (1 divided by the number of conditions when the conditions are balanced, as the order of the images makes them).

Markers: ✅ the conditions can be told apart across repeats; ⚠️ a trend only; ➖ no conclusion possible (too few repeats) or not above chance; ❌ check the data first. A null result is a result.

In [ ]:
# @title Analyse the results
from guessthecondition.report import accuracy_figure, confusion_figure, decision_time_figure, figure_png, write_report

if session is None or session.n_answered == 0:
    print("⚠️ Guess some images first (Steps 1 and 2).")
else:
    answered = session.answered()
    analysis = analyze(session.table)
    print(analysis.text())
    display(Image(figure_png(confusion_figure(analysis))))
    display(Image(figure_png(accuracy_figure(analysis))))
    if len(analysis.decision_time):
        display(Image(figure_png(decision_time_figure(answered))))
    paths = write_report(analysis, answered, session.folder / "analysis")
    print("\n📄 The PDF report, the figures and the tables are in: %s" % paths["analysis_results.pdf"].parent)

# Version log
---
**Version 0.1.0.** First release of the LabConstrictor app. It brings together the Napari plugin and the Google Colab notebook of Guess the Condition:

* one shared package (`guessthecondition`) behind the notebook, the Napari and Fiji tools and the Python API;
* 2D images, multi-channel images and z-stacks are shown correctly;
* the images are spread over the conditions and the repeats, as the first versions announced;
* the answer is read across the biological repeats; the exact number of correct guesses is used in the binomial test;
* guesses can be undone, a game can be resumed from any of the front ends, and a restart keeps a backup.